# Stock Market Regime Detection & Regime-Switching Strategy
### Full End-to-End Walkthrough

This notebook walks through the entire pipeline:
1. Download and explore price data
2. Engineer technical features
3. Detect market regimes with an HMM
4. Train a Random Forest to *predict* regimes
5. Backtest the regime-switching strategy
6. Compare against benchmarks

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from features import fetch_data, build_features
from regime_detection import fit_hmm, plot_regimes, get_regime_labels
from model import prepare_dataset, walk_forward_eval, tune_and_train, evaluate_model
from strategy import (
    backtest_regime_switching, backtest_buy_and_hold,
    backtest_momentum_only, backtest_meanrev_only,
    compute_metrics, plot_backtest
)

TICKER = 'SPY'
START = '2015-01-01'
END   = '2024-01-01'

print('All imports OK!')

## Step 1: Download Data

In [ ]:
df = fetch_data(TICKER, START, END)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['close'], color='#1a1a2e', linewidth=1)
axes[0].set_title(f'{TICKER} Price History', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].grid(alpha=0.3)

log_returns = np.log(df['close'] / df['close'].shift(1)).dropna()
axes[1].fill_between(log_returns.index, log_returns, color='#457b9d', alpha=0.6)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Daily Log Returns', fontsize=12)
axes[1].set_ylabel('Log Return')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nBasic stats:')
print(log_returns.describe())

## Step 2: Feature Engineering

In [ ]:
features = build_features(df)
print('\nFeature matrix shape:', features.shape)
features.head()

## Step 3: Regime Detection via HMM

In [ ]:
returns = np.log(df['close'] / df['close'].shift(1)).dropna()
model_hmm, regimes = fit_hmm(returns, n_states=2)

price_aligned = df['close'].loc[returns.index]
plot_regimes(price_aligned, regimes, ticker=TICKER)

## Step 4: Train the Regime Classifier

In [ ]:
regimes_series = get_regime_labels(df, features.index)
common_idx = features.index.intersection(regimes_series.index)
features_aligned = features.loc[common_idx]
regimes_aligned = regimes_series.loc[common_idx]

X, y = prepare_dataset(features_aligned, regimes_aligned)

# Walk-forward CV
cv_results = walk_forward_eval(X, y)

# Train/test split (temporal)
split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

rf_model, scaler = tune_and_train(X_train, y_train)
preds = evaluate_model(rf_model, scaler, X_test, y_test, feature_names=X.columns.tolist())

## Step 5: Backtest

In [ ]:
X_test_scaled = scaler.transform(X_test)
regime_preds = pd.Series(rf_model.predict(X_test_scaled), index=X_test.index)

test_features = features_aligned.loc[X_test.index]

result_df = backtest_regime_switching(df, test_features, regime_preds)
bh   = backtest_buy_and_hold(df, test_features)
mom  = backtest_momentum_only(df, test_features)
mr   = backtest_meanrev_only(df, test_features)

all_strategies = {
    'Regime-Switching': result_df['equity'],
    'Buy & Hold': bh,
    'Momentum Only': mom,
    'Mean-Reversion Only': mr,
}

plot_backtest(all_strategies, ticker=TICKER)

In [ ]:
print('\n=== Final Performance Summary ===')
for name, equity in all_strategies.items():
    m = compute_metrics(equity)
    print(f'\n{name}')
    for k, v in m.items():
        print(f'  {k}: {v}')